In [2]:
import geopandas as gpd 
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np 
import shapely as shp
import os 
from math import atan, sin, cos
from scipy.stats import binned_statistic_2d

In [3]:
os.chdir('..')
from functions.funcs import *

In [5]:
data = gpd.read_file(r"C:\FATE\Palmyra Data\MI_and_SAT_FAD_positions")
line = data['geometry'][0]
x,y = line.xy


In [ ]:
delx = np.diff(x)
dely = np.diff(y)
direction = np.arctan(dely/delx)
print(np.rad2deg(direction))
print(len(direction))
print(len(delx))
print(len(x))

In [ ]:
line = line ## singular geometry item 
x,y = line.xy
delx = np.diff(x) ## this is one value that needs to be added into an array to latter be stored in pandas data set. 
dely = np.diff(x) ## this is one value that needs to be added into an array to latter be stored in pandas data set. 

In [6]:
delx_list = []
dely_list = []
test_data = data
delx_long = np.array([])
dely_long = np.array([])

lines = test_data["geometry"]
length = lines.shape[0]

delxs = []
delys=[]

# bigger loop 

##for id in data['BuoyName']:
for n in range(0,length):
    singleline = lines[n]
    coords = list(lines[n].coords)

    x_coords = []
    y_coords = []
    for x, y in coords:
        x_coords.append(x)
        y_coords.append(y)
    #for x_c in x_coords:
    delx = np.diff(x_coords)
    #for y_c in y_coords:
    dely=np.diff(y_coords)

    delxs.append(delx)
    delys.append(dely)

    delx_long = np.append(delx_long, delx)
    dely_long = np.append(dely_long, dely)

data["x"] = delxs
data["y"] = delys
print(dely_long.shape[0])
print(type(dely_long))
heading = np.atan(delx_long/dely_long)
delx_long = np.cos(heading)
dely_long = np.sin(heading)
print(delx_long[:5])
print(dely_long[:5])

102874
<class 'numpy.ndarray'>
[0.51385521 0.40036765 0.30776368 0.02310779 0.14997912]
[-0.85787693 -0.9163546   0.95146283  0.99973298 -0.98868916]


C:\Users\czerfass\AppData\Local\Temp\1\ipykernel_19312\1913035330.py:40: RuntimeWarning: divide by zero encountered in divide
  heading = np.atan(delx_long/dely_long)


In [ ]:
##need list of all lat, lon and, directions (size of these are going to be differnt, have to shorten the last lat, lon point)

lines = data["geometry"]

points = np.empty((1,2))
for n in range(0,len(lines)):
    dummyline = lines[n]
    cords  = np.array(dummyline.coords)
    cords = cords[:-1]
    points = np.append(points,cords, axis= 0 )

rotatedpoints = np.rot90(points) 
print(rotatedpoints.shape) ##this gives us all of lats in first row and all of the lon in second row 

lat  = rotatedpoints[0][1:]
lon = rotatedpoints[1][1:]


In [ ]:
x_values = binned_statistic_2d(lon, lat, delx_long, statistic= "mean", bins = 20)
y_values = binned_statistic_2d(lon, lat, dely_long, statistic= "mean", bins = 20)
xbound = x_values[1]
ybound = x_values[2]
X, Y = np.meshgrid(xbound[:-1], ybound[:-1])
print(X.shape)
print(Y.shape)
print(x_values[0].shape)
print(y_values[0].shape )

In [ ]:
fig, ax = plt.subplots()
ax.quiver(X, Y,x_values[0],y_values[0])
ax.set_title("inncorect plot, (this is closer to speed)")
ax.set_xlabel("longitude")
ax.set_ylabel("Latitude")
fig.savefig(r"C:\FATE\Figures\Direction Vectors_Incorrect.png")